In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

import os
import argparse
from pathlib import Path
import sentencepiece as spm
from datasets import load_dataset, DatasetDict
from transformers import (
    PreTrainedTokenizerFast,
    GPTNeoXConfig,
    GPTNeoXForCausalLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
)
from tqdm import tqdm


def train_sentencepiece(corpus_path: str, sp_prefix: str, vocab_size: int,
                        character_coverage: float = 1.0, model_type: str = "unigram",
                        input_sentence_size: int = 0, shuffle_input_sentence: bool = True):
    """
    Treina um SentencePiece "normal" (subword). Não colapsa frase=token.
    O objetivo é o modelo aprender conteúdo interno das frases.
    """
    spm.SentencePieceTrainer.Train(
        input=corpus_path,
        model_prefix=sp_prefix,
        vocab_size=vocab_size,
        character_coverage=character_coverage,
        model_type=model_type,
        input_sentence_size=input_sentence_size,
        shuffle_input_sentence=shuffle_input_sentence,
        unk_id=0,
        bos_id=1,
        eos_id=2,
        pad_id=3,
        hard_vocab_limit=False,  # evita erro se faltar uns poucos símbolos
    )
    print(f"[SPM] Treinado em {sp_prefix}.model / {sp_prefix}.vocab")


def build_tokenizer(sp_model_path: str):
    tokenizer = PreTrainedTokenizerFast(
        tokenizer_file=sp_model_path,
        bos_token="<s>",
        eos_token="</s>",
        unk_token="<unk>",
        pad_token="<pad>",
    )
    # adiciona o delimitador de fim de frase
    special_tokens_dict = {"additional_special_tokens": ["<SENT_END>"]}
    tokenizer.add_special_tokens(special_tokens_dict)
    print("[TOK] Adicionado token especial <SENT_END> (id =", tokenizer.convert_tokens_to_ids("<SENT_END>"), ")")
    return tokenizer


def prepare_stream_dataset(input_path: str, tokenizer, block_size: int):
    """
    Lê arquivo linha-a-linha; injeta <SENT_END> ao final de cada linha;
    tokeniza; concatena e cria blocos fixos (causal LM).
    """
    raw = load_dataset("text", data_files={"train": input_path})

    SENT_END = "<SENT_END>"

    def add_delim(example):
        txt = (example["text"] or "").strip()
        if not txt:
            return {"text": ""}
        return {"text": f"{txt} {SENT_END} "}

    with_delim = raw.map(add_delim)

    def tok_fn(batch):
        return tokenizer(batch["text"])

    tok = with_delim["train"].map(tok_fn, batched=True, remove_columns=["text"])

    def group_texts(examples):
        concatenated = []
        for seq in examples["input_ids"]:
            concatenated.extend(seq)
        total_len = (len(concatenated) // block_size) * block_size
        concatenated = concatenated[:total_len]
        input_ids = [concatenated[i:i+block_size] for i in range(0, total_len, block_size)]
        attention_mask = [[1]*len(x) for x in input_ids]
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": [ids.copy() for ids in input_ids],
        }

    lm_ds = tok.map(group_texts, batched=True, batch_size=1000)
    return DatasetDict({"train": lm_ds})


def build_model(tokenizer, hidden_size: int, n_layers: int, n_heads: int, intermediate_size: int,
                max_position_embeddings: int):
    config = GPTNeoXConfig(
        vocab_size=len(tokenizer),
        hidden_size=hidden_size,
        num_hidden_layers=n_layers,
        num_attention_heads=n_heads,
        intermediate_size=intermediate_size,
        max_position_embeddings=max_position_embeddings,
    )
    model = GPTNeoXForCausalLM(config)
    # importante se você carregar um modelo pré-treinado: model.resize_token_embeddings(len(tokenizer))
    return model


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--input_file", required=True, help="Arquivo com 1 frase por linha (logs etc.)")
    ap.add_argument("--out_dir", default="./neox_sent", help="Saída (modelo/tokenizer/resultados)")
    ap.add_argument("--vocab_size", type=int, default=8000, help="Vocab do SentencePiece (subword)")
    ap.add_argument("--block_size", type=int, default=512, help="Comprimento dos blocos de treino")
    ap.add_argument("--epochs", type=int, default=3)
    ap.add_argument("--batch_size", type=int, default=8)
    ap.add_argument("--lr", type=float, default=5e-4)
    ap.add_argument("--weight_decay", type=float, default=0.01)
    ap.add_argument("--hidden_size", type=int, default=512)
    ap.add_argument("--n_layers", type=int, default=8)
    ap.add_argument("--n_heads", type=int, default=8)
    ap.add_argument("--intermediate_size", type=int, default=2048)
    ap.add_argument("--max_pos", type=int, default=2048)
    ap.add_argument("--fp16", action="store_true")
    args = ap.parse_args()

    out = Path(args.out_dir)
    out.mkdir(parents=True, exist_ok=True)

    sp_prefix = str(out / "sp_logs")
    if not Path(sp_prefix + ".model").exists():
        train_sentencepiece(
            corpus_path=args.input_file,
            sp_prefix=sp_prefix,
            vocab_size=args.vocab_size,
            character_coverage=1.0,
            model_type="unigram",
            input_sentence_size=0,
            shuffle_input_sentence=True,
        )

    tokenizer = build_tokenizer(sp_prefix + ".model")

    # dataset
    ds = prepare_stream_dataset(args.input_file, tokenizer, block_size=args.block_size)

    # modelo
    model = build_model(
        tokenizer,
        hidden_size=args.hidden_size,
        n_layers=args.n_layers,
        n_heads=args.n_heads,
        intermediate_size=args.intermediate_size,
        max_position_embeddings=args.max_pos,
    )

    # data collator e argumentos de treino
    collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    training_args = TrainingArguments(
        output_dir=str(out / "ckpts"),
        per_device_train_batch_size=args.batch_size,
        gradient_accumulation_steps=1,
        learning_rate=args.lr,
        num_train_epochs=args.epochs,
        weight_decay=args.weight_decay,
        logging_steps=50,
        save_steps=2000,
        save_total_limit=2,
        report_to="none",
        fp16=args.fp16,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=ds["train"],
        data_collator=collator,
        tokenizer=tokenizer,
    )

    trainer.train()

    # salva
    save_dir = out / "final"
    save_dir.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(save_dir))
    tokenizer.save_pretrained(str(save_dir))
    print(f"[OK] Modelo e tokenizer salvos em: {save_dir}")


if __name__ == "__main__":
    main()

sentencepiece_trainer.cc(77) LOG(INFO) Starts training with : 
trainer_spec {
  input: logs/part_3.log
  input_format: 
  model_prefix: sp_logs
  model_type: UNIGRAM
  vocab_size: 8000
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differential_privacy: 0
  differential_privacy_noise_level: 0
 

RuntimeError: Internal: src/trainer_interface.cc(661) [(trainer_spec_.vocab_size()) == (model_proto->pieces_size())] Vocabulary size too high (8000). Please set it to a value <= 403.